# IMDB Sentiment Classification — Improved Pipeline

Improvements over the baseline (BERT-Tiny, val acc ~0.90):

1. **Mean-pooling head** instead of `[CLS]` (small BERTs have weak `[CLS]` representations).
2. **Layer-wise LR decay** + lower base LR + longer warmup.
3. **Dynamic padding** (collator pads to longest in batch) with `MAX_LEN=320`.
4. **Iterative pseudo-labeling** with 3 rounds and balanced classes.
5. **Test-time augmentation** (head-of-review + tail-of-review).
6. **Multi-seed ensembling** (3 seeds, average probabilities).
7. **Final retrain on full labeled set** (train + val combined).
8. **Better text cleaning** (HTML entities, repeated punctuation).

Stays under the **10M parameter** budget (BERT-Tiny: 9,591,554 params).

## 1. Setup

In [ ]:
from pathlib import Path
import random
import re
import html
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Load and clean data

The cleaner now also unescapes HTML entities (`&quot;`, `&amp;`, etc.) and collapses repeated punctuation.

In [ ]:
DATA_DIR = Path("imdb-review-classification")

print("Using data folder:", DATA_DIR.resolve())

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

def clean_review(text):
    text = str(text)
    # Strip HTML tags (most commonly <br />)
    text = re.sub(r'<[^>]+>', ' ', text)
    # Decode HTML entities (&quot;, &amp;, &#39;, etc.)
    text = html.unescape(text)
    # Collapse 3+ repeated punctuation: "!!!!!" -> "!!"
    text = re.sub(r'([!?.,])\1{2,}', r'\1\1', text)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

train_df["review"] = train_df["review"].apply(clean_review)
test_df["review"]  = test_df["review"].apply(clean_review)

train_df.columns = train_df.columns.str.strip().str.lower()
test_df.columns  = test_df.columns.str.strip().str.lower()
train_df["label"] = train_df["label"].astype(str).str.strip().str.lower()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print(train_df["label"].value_counts())

## 3. Tokenizer & dataset

Key change: dataset returns raw token ids without padding. Padding happens **per-batch** in the collator ("dynamic padding"), which is much faster and gives slightly better gradients than padding everything to 512.

In [ ]:
MODEL_NAME = "google/bert_uncased_L-2_H-256_A-4"  # BERT-Tiny: 9.59M params

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
MAX_LEN = 400  # shorter than 512: most reviews fit, faster training

print("Pretrained model:", MODEL_NAME)
print("Tokenizer vocab size:", len(tokenizer))

label_to_num = {"negative": 0, "positive": 1}
num_to_label = {0: "negative", 1: "positive"}


class IMDBDataset(Dataset):
    def __init__(self, df, tokenizer, has_labels=True, max_len=MAX_LEN, mode="head"):
        """
        mode='head': keep first max_len tokens (default).
        mode='tail': keep last max_len tokens (used at inference for TTA).
        """
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.has_labels = has_labels
        self.max_len = max_len
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        review = str(self.df.loc[idx, "review"])

        if self.mode == "tail":
            # Tokenize without truncation, then keep the LAST max_len tokens.
            full = self.tokenizer(review, add_special_tokens=False)["input_ids"]
            keep = full[-(self.max_len - 2):]  # leave room for [CLS] and [SEP]
            ids = [self.tokenizer.cls_token_id] + keep + [self.tokenizer.sep_token_id]
            attn = [1] * len(ids)
        else:
            enc = self.tokenizer(
                review,
                max_length=self.max_len,
                truncation=True,
                padding=False,
                return_attention_mask=True,
            )
            ids = enc["input_ids"]
            attn = enc["attention_mask"]

        item = {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
        }
        if self.has_labels:
            label_text = str(self.df.loc[idx, "label"]).strip().lower()
            item["labels"] = torch.tensor(label_to_num[label_text], dtype=torch.long)
        return item


def collate_fn(batch):
    """Dynamic padding: pad each batch to the longest sequence in that batch."""
    max_len = max(item["input_ids"].size(0) for item in batch)
    pad_id = tokenizer.pad_token_id

    input_ids, attn_masks, labels = [], [], []
    for item in batch:
        L = item["input_ids"].size(0)
        pad = max_len - L
        input_ids.append(F.pad(item["input_ids"], (0, pad), value=pad_id))
        attn_masks.append(F.pad(item["attention_mask"], (0, pad), value=0))
        if "labels" in item:
            labels.append(item["labels"])

    out = {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attn_masks),
    }
    if labels:
        out["labels"] = torch.stack(labels)
    return out

## 4. Model: BERT-Tiny + mean-pooling head

`AutoModelForSequenceClassification` uses the `[CLS]` token, which is poorly trained in BERT-Tiny.
We replace it with **mean-pooling** over the last hidden state (masked by attention), which gives
a stronger sentence representation in tiny models. Empirically this adds ~0.5–1% on IMDB.

Total params stay at ~9.59M (the head changes are tiny).

In [ ]:
class MeanPoolBertClassifier(nn.Module):
    def __init__(self, model_name, num_labels=2, dropout=0.2):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        last = out.last_hidden_state                       # (B, L, H)
        mask = attention_mask.unsqueeze(-1).float()        # (B, L, 1)
        summed = (last * mask).sum(dim=1)                  # (B, H)
        counts = mask.sum(dim=1).clamp(min=1e-9)           # (B, 1)
        pooled = summed / counts                           # (B, H)
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
        return {"loss": loss, "logits": logits}


# Sanity check: parameter count
_tmp = MeanPoolBertClassifier(MODEL_NAME)
n_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
assert n_params < 10_000_000, "Over 10M parameter budget!"
del _tmp

## 5. Training utilities with layer-wise LR decay

Layer-wise LR decay applies a smaller learning rate to lower (earlier) transformer layers
and a larger one to upper layers and the head. This is the standard recipe for fine-tuning
BERT-family models and helps prevent the early layers from forgetting their pretrained
knowledge.

In [ ]:
def get_grouped_parameters(model, base_lr, layer_decay=0.85, weight_decay=0.01):
    """
    Layer-wise LR decay: head gets base_lr, then each layer down gets multiplied by layer_decay.
    Embeddings get the smallest LR.
    """
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    n_layers = model.backbone.config.num_hidden_layers  # 2 for BERT-Tiny

    # Group "depth" levels: 0 = embeddings, 1..n_layers = transformer layers,
    # n_layers+1 = head (classifier + dropout)
    def get_depth(name):
        if name.startswith("backbone.embeddings"):
            return 0
        if name.startswith("backbone.encoder.layer."):
            layer_idx = int(name.split("backbone.encoder.layer.")[1].split(".")[0])
            return layer_idx + 1
        # pooler (unused), classifier, etc. -> top
        return n_layers + 1

    groups = {}
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        depth = get_depth(name)
        # higher depth -> closer to base_lr, lower depth -> smaller lr
        lr = base_lr * (layer_decay ** (n_layers + 1 - depth))
        wd = 0.0 if any(nd in name for nd in no_decay) else weight_decay
        key = (lr, wd)
        groups.setdefault(key, []).append(param)

    return [{"params": params, "lr": lr, "weight_decay": wd} for (lr, wd), params in groups.items()]


def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**batch)
        loss = out["loss"]
        logits = out["logits"]

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        bs = batch["labels"].size(0)
        total_loss += loss.item() * bs
        preds = logits.argmax(dim=1)
        correct += (preds == batch["labels"]).sum().item()
        total += bs
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        loss = out["loss"]
        logits = out["logits"]
        bs = batch["labels"].size(0)
        total_loss += loss.item() * bs
        preds = logits.argmax(dim=1)
        correct += (preds == batch["labels"]).sum().item()
        total += bs
    return total_loss / total, correct / total


def train_model(
    train_df_in,
    val_df_in,
    seed=SEED,
    epochs=8,
    batch_size=64,
    base_lr=3e-5,
    warmup_ratio=0.15,
    layer_decay=0.85,
    patience=3,
    save_path="best_model.pt",
    verbose=True,
):
    set_seed(seed)

    train_ds = IMDBDataset(train_df_in, tokenizer, has_labels=True)
    val_ds   = IMDBDataset(val_df_in,   tokenizer, has_labels=True)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=0)

    model = MeanPoolBertClassifier(MODEL_NAME).to(device)
    optimizer = torch.optim.AdamW(get_grouped_parameters(model, base_lr, layer_decay))
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * warmup_ratio),
        num_training_steps=total_steps,
    )

    best_val_acc = 0.0
    bad_epochs = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        va_loss, va_acc = evaluate(model, val_loader, device)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)

        if verbose:
            print(f"[seed={seed}] Epoch {epoch}/{epochs} | "
                  f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
                  f"Val Loss: {va_loss:.4f} Acc: {va_acc:.4f}")

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            bad_epochs = 0
            torch.save(model.state_dict(), save_path)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch}")
                break

    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()
    return model, best_val_acc, history

## 6. Train initial model (single seed, with val split)

We train one model first to use as the pseudo-labeling teacher.

In [ ]:
train_data, val_data = train_test_split(
    train_df, test_size=0.2, random_state=SEED, stratify=train_df["label"]
)
print("Train split:", train_data.shape)
print("Val split:  ", val_data.shape)

model, best_val_acc, history = train_model(
    train_data,
    val_data,
    seed=SEED,
    epochs=20,
    batch_size=64,
    base_lr=3e-5,
    warmup_ratio=0.15,
    layer_decay=0.85,
    patience=4,
    save_path="best_initial.pt",
)

print(f"Initial best val accuracy: {best_val_acc:.4f}")

In [ ]:
# Plot training curves
epochs_ran = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.semilogy(epochs_ran, history["train_loss"], label="Train")
plt.semilogy(epochs_ran, history["val_loss"], label="Val")
plt.title("Loss"); plt.xlabel("Epoch"); plt.legend(); plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(epochs_ran, history["train_acc"], label="Train")
plt.title("Train Accuracy"); plt.xlabel("Epoch"); plt.ylim([0.5, 1.0]); plt.grid(True); plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs_ran, history["val_acc"], label="Val")
plt.title("Val Accuracy"); plt.xlabel("Epoch"); plt.ylim([0.5, 1.0]); plt.grid(True); plt.legend()

plt.tight_layout(); plt.show()

## 7. Iterative pseudo-labeling

Instead of one round of pseudo-labeling, we do **3 rounds** with a gradually loosening confidence threshold.
Each round:
1. Score the unlabeled `aclImdb/train/unsup` reviews with the current best model.
2. Keep only confident predictions.
3. **Balance** the kept pseudo-labels (equal positives and negatives) — your original code had a 24k vs 21k imbalance.
4. Combine with original labeled training data and retrain.

Skip this section if you don't have the unlabeled `aclImdb` data.

In [ ]:
UNSUP_DIR = Path("aclImdb/train/unsup")

if UNSUP_DIR.exists():
    unlabeled_reviews = [f.read_text(encoding="utf-8") for f in UNSUP_DIR.glob("*.txt")]
    unlabeled_df = pd.DataFrame({"review": unlabeled_reviews})
    unlabeled_df["review"] = unlabeled_df["review"].apply(clean_review)
    print(f"Loaded {len(unlabeled_df)} unlabeled reviews")
    HAS_UNSUP = True
else:
    print("No unsup data found, skipping pseudo-labeling.")
    HAS_UNSUP = False

In [ ]:
@torch.no_grad()
def score_unlabeled(model, df_unlab, batch_size=128):
    """Return predicted probability of POSITIVE class for each row."""
    ds = IMDBDataset(df_unlab, tokenizer, has_labels=False)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    all_probs = []
    model.eval()
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        probs = F.softmax(out["logits"], dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_probs)


def make_pseudo_labels(probs_pos, df_unlab, hi=0.97, lo=0.03, balance=True):
    """Pick confident pseudo-labels and (optionally) class-balance them."""
    pos_mask = probs_pos >= hi
    neg_mask = probs_pos <= lo

    pos_idx = np.where(pos_mask)[0]
    neg_idx = np.where(neg_mask)[0]

    if balance:
        n = min(len(pos_idx), len(neg_idx))
        rng = np.random.RandomState(SEED)
        pos_idx = rng.choice(pos_idx, n, replace=False)
        neg_idx = rng.choice(neg_idx, n, replace=False)

    keep_idx = np.concatenate([pos_idx, neg_idx])
    pseudo = df_unlab.iloc[keep_idx].copy().reset_index(drop=True)
    pseudo["label"] = ["positive"] * len(pos_idx) + ["negative"] * len(neg_idx)
    pseudo["id"] = -1  # sentinel: not a real id
    return pseudo[["id", "review", "label"]]

In [ ]:
if HAS_UNSUP:
    teacher_model = model
    current_train_df = train_df.copy()  # full 45k labeled

    rounds = [
        {"hi": 0.97, "lo": 0.03, "epochs": 20,  "lr": 3e-5},
        {"hi": 0.95, "lo": 0.05, "epochs": 20,  "lr": 2e-5},
        {"hi": 0.92, "lo": 0.08, "epochs": 20,  "lr": 2e-5},
        {"hi": 0.91, "lo": 0.11, "epochs": 20,  "lr": 2e-5},
        {"hi": 0.9, "lo": 0.1, "epochs": 20,  "lr": 1e-5},
    ]

    for r_idx, cfg in enumerate(rounds, start=1):
        print(f"\n=== Pseudo-labeling round {r_idx}: hi={cfg['hi']}, lo={cfg['lo']} ===")
        probs_pos = score_unlabeled(teacher_model, unlabeled_df)
        pseudo_df = make_pseudo_labels(probs_pos, unlabeled_df, hi=cfg["hi"], lo=cfg["lo"], balance=True)
        print(f"Kept {len(pseudo_df)} pseudo-labeled (balanced) | "
              f"pos={(pseudo_df['label']=='positive').sum()} "
              f"neg={(pseudo_df['label']=='negative').sum()}")

        # Combine original labeled + pseudo-labeled
        combined = pd.concat([train_df, pseudo_df], ignore_index=True)
        train_data_r, val_data_r = train_test_split(
            combined, test_size=0.1, random_state=SEED + r_idx, stratify=combined["label"]
        )

        teacher_model, va_acc, _hist = train_model(
            train_data_r,
            val_data_r,
            seed=SEED + r_idx,
            epochs=cfg["epochs"],
            batch_size=64,
            base_lr=cfg["lr"],
            warmup_ratio=0.1,
            layer_decay=0.85,
            patience=3,
            save_path=f"best_pseudo_round{r_idx}.pt",
        )
        print(f"Round {r_idx} best val acc: {va_acc:.4f}")

    model = teacher_model  # use the final pseudo-trained model going forward

## 8. Multi-seed ensemble (final stage)

Train 3 models with different seeds on the **full 45k labeled training set + balanced pseudo-labels** (no val split — we already chose hyperparameters above). Average their test probabilities.

If you didn't run pseudo-labeling, this still gives a meaningful boost from seed averaging alone.

In [ ]:
# Build the final training set: full labeled + (optionally) confident pseudo-labels
if HAS_UNSUP:
    probs_pos_final = score_unlabeled(model, unlabeled_df)
    final_pseudo = make_pseudo_labels(
        probs_pos_final, unlabeled_df, hi=0.95, lo=0.05, balance=True
    )
    full_train_df = pd.concat([train_df, final_pseudo], ignore_index=True)
    print(f"Final training set: {len(full_train_df)} "
          f"({len(train_df)} labeled + {len(final_pseudo)} pseudo)")
else:
    full_train_df = train_df.copy()
    print(f"Final training set: {len(full_train_df)} labeled")

In [ ]:
SEEDS = [42, 2005, 7, 0, 2026]
NUM_EPOCHS_FINAL = 20  # roughly the best_epoch we observed earlier

ensemble_models = []

for s in SEEDS:
    print(f"\n=== Training ensemble member, seed={s} ===")
    set_seed(s)

    # Tiny holdout (5%) just for early stopping; we mostly train on everything
    tr, va = train_test_split(full_train_df, test_size=0.05, random_state=s, stratify=full_train_df["label"])

    m, va_acc, _h = train_model(
        tr,
        va,
        seed=s,
        epochs=NUM_EPOCHS_FINAL,
        batch_size=64,
        base_lr=2e-5,
        warmup_ratio=0.1,
        layer_decay=0.85,
        patience=3,
        save_path=f"ensemble_seed{s}.pt",
    )
    ensemble_models.append(m)
    print(f"Seed {s}: best val acc = {va_acc:.4f}")

## 9. Test-time augmentation + ensemble inference

For each test review, we score it twice — once using the **first** 320 tokens and once using the **last** 320 tokens — for each of the 3 ensemble models. We average all 6 probabilities to get the final prediction.

Long reviews often state the verdict at the end (e.g., "...overall, I really enjoyed it"), so the tail-truncation gives a complementary view.

In [ ]:
@torch.no_grad()
def get_probs_tta(models, df, batch_size=128):
    """Return averaged positive-class probability across models and head/tail crops."""
    all_runs = []
    for m in models:
        m.eval()
        for mode in ("head", "tail"):
            ds = IMDBDataset(df, tokenizer, has_labels=False, mode=mode)
            loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
            run_probs = []
            for batch in loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                out = m(**batch)
                probs = F.softmax(out["logits"], dim=1)[:, 1]
                run_probs.extend(probs.cpu().numpy())
            all_runs.append(np.array(run_probs))
    return np.mean(all_runs, axis=0)  # average over (n_models * 2) runs

## 10. Generate submission

In [ ]:
test_probs = get_probs_tta(ensemble_models, test_df, batch_size=128)
test_preds = (test_probs >= 0.5).astype(int)

submission = pd.DataFrame({
    "Id": test_df["id"].values,
    "Label": [num_to_label[p] for p in test_preds],
})

submission.to_csv("prediction.csv", index=False)
print("Wrote prediction.csv")
print(submission.head())
print()
print("Predicted label distribution:")
print(submission["Label"].value_counts())

## Notes on what to tune if you want to go further

- **Longer pseudo-labeling rounds.** If you have GPU time, run 4–5 rounds instead of 3, and keep more samples each time.
- **More ensemble members.** 5–7 seeds is better than 3; the gains taper off but don't disappear.
- **MAX_LEN.** Try 384 or 256 — there's no universally best choice on IMDB.
- **Stronger backbone.** If your project's parameter rule counts only **trainable** parameters, you can use **BERT-Mini** (`google/bert_uncased_L-4_H-256_A-4`, 11.2M total) and freeze the embedding table — that drops trainable to **~3.2M** and typically adds another +1–2% accuracy thanks to the deeper encoder. Just change `MODEL_NAME` and add `for p in model.backbone.embeddings.parameters(): p.requires_grad_(False)` in the model constructor. Verify your project's wording before doing this.